### 1. Data loading

In [1]:
from rdkit.Chem import AllChem as chem
from rdkit import Chem as c
import rdkit.DataStructs as data
from rdkit.Chem import Descriptors3D

import os.path as osp
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import json

data_folder = 'dataset/mossbauer/raw'
smiles_csv = pd.read_csv(osp.join(data_folder, 'annotations.tsv'),sep='\t')

file_list = list(smiles_csv.iloc[:,0])
mol_list = []
density_list = []
quadrupole_list = []

for file in file_list:
    file_path = None
    if(osp.exists((file_path := osp.join(data_folder, file[:-4], 'xtbtopo.mol')))):
        pass
    else:
        file_path = osp.join(data_folder, file[:-4], 'xtbopt.mol')

    desc_path = osp.join(data_folder, file[:-4], 'tblite-data.npz')
    post_processing = np.load(desc_path)

    mol = chem.MolFromMolFile(file_path, sanitize=False, removeHs=False)
    
    # Regenerate computed properties like implicit valence and ring information
    mol.UpdatePropertyCache(strict=False)

    # Apply several sanitization rules
    chem.SanitizeMol(mol,c.SanitizeFlags.SANITIZE_FINDRADICALS| \
                         c.SanitizeFlags.SANITIZE_SETAROMATICITY| \
                         c.SanitizeFlags.SANITIZE_SETCONJUGATION| \
                         c.SanitizeFlags.SANITIZE_SETHYBRIDIZATION| \
                         c.SanitizeFlags.SANITIZE_SYMMRINGS,catchErrors=True)

    iron_index = 0
    for i, a in enumerate(mol.GetAtoms()):
        if a.GetAtomicNum() == 26:
            iron_index = i
            break

    density = [post_processing['tblite0_q_A_alpha'][iron_index] + post_processing['tblite0_q_A_beta'][iron_index],
               post_processing['tblite0_p_s_alpha'][iron_index],
               post_processing['tblite0_p_p_alpha'][iron_index],
               post_processing['tblite0_p_d_alpha'][iron_index],
               post_processing['tblite0_p_s_beta'][iron_index],
               post_processing['tblite0_p_p_beta'][iron_index],
               post_processing['tblite0_p_d_beta'][iron_index],
               post_processing['tblite0_chem_pot_alpha'][iron_index],
               post_processing['tblite0_chem_pot_beta'][iron_index],
               post_processing['tblite0_gap_alpha'][iron_index],
               post_processing['tblite0_gap_beta'][iron_index],
               post_processing['tblite0_HOAO_alpha'][iron_index],
               post_processing['tblite0_HOAO_beta'][iron_index],
               post_processing['tblite0_LUAO_alpha'][iron_index],
               post_processing['tblite0_LUAO_beta'][iron_index]]

    quadrupole = [post_processing[f'tblite0_qm_A_{x+13*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_qm_A_{x+13*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_ext_qm_e_{x+9*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_ext_qm_e_{x+9*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_qm_A_alpha'][iron_index], 
                  post_processing[f'tblite0_qm_A_beta'][iron_index],
                  post_processing[f'tblite0_ext_qm_e_alpha'][iron_index],
                  post_processing[f'tblite0_ext_qm_e_beta'][iron_index],
                  post_processing[f'tblite0_ext_qm_Z_alpha'][iron_index],
                  post_processing[f'tblite0_ext_qm_Z_beta'][iron_index]]

    density_list.append(density)
    quadrupole_list.append(quadrupole)
    
    mol_list.append(mol)

rdkit_gen = chem.GetRDKitFPGenerator(fpSize=2048, maxPath=7)

fp_list = []
descriptor_3d_list = []
for mol in mol_list:
    fp = np.zeros((0,))

    fpgen = rdkit_gen.GetCountFingerprint(mol)
    data.ConvertToNumpyArray(fpgen, fp)
    fp_list.append(fp)
    
    descriptor_3d_list.append(np.asarray(list(Descriptors3D.CalcMolDescriptors3D(mol).values())))
    
fp_list = np.asarray(fp_list)

IS = np.asarray(smiles_csv.iloc[:, 1])
QS = np.asarray(smiles_csv.iloc[:, 2])

TEMP = np.asarray([smiles_csv.loc[:, 'TEMP']])
spin = np.asarray(smiles_csv.loc[:, 'SPIN'])
fespin = np.asarray(smiles_csv.loc[:, 'FE SPIN'])
charge = np.asarray(smiles_csv.loc[:, 'CHARGE'])

spin_one_hot = np.eye(6)[(spin * 2).astype(int)]
fe_spin_one_hot = np.eye(6)[(fespin * 2).astype(int)]
charge_one_hot = np.eye(8)[charge + 4]

X = np.concatenate((fp_list, descriptor_3d_list, TEMP.T, spin_one_hot, fe_spin_one_hot, charge_one_hot, quadrupole_list, density_list), axis=1)
Y = IS

with open('split.json', 'r') as file:
    split = json.load(file)

train_X = X[split['train']]
train_y = Y[split['train']]
val_X = X[split['validation']]
val_y = Y[split['validation']]
test_X = X[split['test']]
test_y = Y[split['test']]

datasets_X = [train_X, val_X, test_X]
datasets_y = [train_y, val_y, test_y]

from rdkit.Chem import Descriptors3D

import pandas as pd
import numpy as np
from pathlib import Path

from lightning import pytorch as pl

from chemprop import data, featurizers, models, nn

num_workers = 10 # number of workers for dataloader. 0 means using main process for data loading
target_columns = ['IS', 'QS'] # classification of activity (either 0 or 1)
feature_columns = ['CHARGE', 'SPIN', 'FE SPIN', 'TEMP'] # name of feature columns in csv

ys = smiles_csv.loc[:, target_columns].values
x_ds = smiles_csv.loc[:, feature_columns].values
x_dsize = 0

datapoint_list = []
for mol, file, y, x_d in zip(mol_list, file_list, ys, x_ds):
    desc_path = osp.join(data_folder, file[:-4], 'tblite-data.npz')
    post_processing = np.load(desc_path)

    charges = post_processing['tblite0_q_A_alpha'] + post_processing['tblite0_q_A_beta']

    d_density_alpha = post_processing['tblite0_p_d_alpha']
    p_density_alpha = post_processing['tblite0_p_p_alpha']
    s_density_alpha = post_processing['tblite0_p_s_alpha']

    d_density_beta = post_processing['tblite0_p_d_beta']
    p_density_beta = post_processing['tblite0_p_p_beta']
    s_density_beta = post_processing['tblite0_p_s_beta']

    chem_pot_alpha = post_processing['tblite0_chem_pot_alpha']
    chem_pot_beta = post_processing['tblite0_chem_pot_beta']
    
    gap_alpha = post_processing['tblite0_gap_alpha']
    gap_beta = post_processing['tblite0_gap_beta']

    hoao_a = post_processing['tblite0_HOAO_alpha']
    hoao_b = post_processing['tblite0_HOAO_beta']

    luao_a = post_processing['tblite0_LUAO_alpha']
    luao_b = post_processing['tblite0_LUAO_beta']

    V_f_final = np.column_stack((charges,
                                 d_density_alpha, p_density_alpha, s_density_alpha, 
                                 d_density_beta, p_density_beta, s_density_beta,
                                 chem_pot_alpha, chem_pot_beta, 
                                 gap_alpha, gap_beta,
                                 hoao_a, hoao_b, luao_a, luao_b))
    
    charge_one_hot = list(np.eye(8)[int(x_d[0] + 4)])
    spin_one_hot = list(np.eye(6)[int(x_d[1] * 2)])
    fe_spin_one_hot = list(np.eye(6)[int(x_d[2] * 2)])

    iron_index = 0
    for i, a in enumerate(mol.GetAtoms()):
        if a.GetAtomicNum() == 26:
            iron_index = i
            break
            
    quadrupole = [post_processing[f'tblite0_qm_A_{x+13*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_qm_A_{x+13*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_ext_qm_e_{x+9*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_ext_qm_e_{x+9*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_alpha'][iron_index] + 
                  post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_beta'][iron_index] 
                  for x in ['zz','yy','xx','xy','xz','yz']] + \
                 [post_processing[f'tblite0_qm_A_alpha'][iron_index], 
                  post_processing[f'tblite0_qm_A_beta'][iron_index],
                  post_processing[f'tblite0_ext_qm_e_alpha'][iron_index],
                  post_processing[f'tblite0_ext_qm_e_beta'][iron_index],
                  post_processing[f'tblite0_ext_qm_Z_alpha'][iron_index],
                  post_processing[f'tblite0_ext_qm_Z_beta'][iron_index]]

    threed_list = list(Descriptors3D.CalcMolDescriptors3D(mol).values())
    
    x_d_final = np.asarray(charge_one_hot + spin_one_hot + fe_spin_one_hot + threed_list + quadrupole + [x_d[3], charges[iron_index]])
    
    x_dsize = x_d_final.shape[0]
    datapoint_list.append(data.MoleculeDatapoint(mol, y, x_d=x_d_final, V_f=V_f_final, name=file))

all_data = np.asarray(datapoint_list)

### 2. Validation study

In [ ]:
import torch
torch.set_float32_matmul_precision('high')
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

from chemprop.nn.transforms import ScaleTransform, GraphTransform, UnscaleTransform

from featurizers.atom import TMCAtomFeaturizer
from featurizers.bond import TMCBondFeaturizer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from rdkit.Chem import Descriptors3D

import pandas as pd
import numpy as np
from pathlib import Path

from lightning import pytorch as pl

from chemprop import data, featurizers, models, nn

with open("scores_k-fold.txt", "w") as f:

    for j in range(1,11):
        
        num_workers = 10 # number of workers for dataloader. 0 means using main process for data loading
        target_columns = ['IS', 'QS'] # classification of activity (either 0 or 1)
        feature_columns = ['CHARGE', 'SPIN', 'FE SPIN', 'TEMP'] # name of feature columns in csv
        
        ys = smiles_csv.loc[:, target_columns].values
        x_ds = smiles_csv.loc[:, feature_columns].values
        x_dsize = 0
        
        datapoint_list = []
        for mol, file, y, x_d in zip(mol_list, file_list, ys, x_ds):
            desc_path = osp.join(data_folder, file[:-4], 'tblite-data.npz')
            post_processing = np.load(desc_path)
        
            charges = post_processing['tblite0_q_A_alpha'] + post_processing['tblite0_q_A_beta']
        
            d_density_alpha = post_processing['tblite0_p_d_alpha']
            p_density_alpha = post_processing['tblite0_p_p_alpha']
            s_density_alpha = post_processing['tblite0_p_s_alpha']
        
            d_density_beta = post_processing['tblite0_p_d_beta']
            p_density_beta = post_processing['tblite0_p_p_beta']
            s_density_beta = post_processing['tblite0_p_s_beta']
        
            chem_pot_alpha = post_processing['tblite0_chem_pot_alpha']
            chem_pot_beta = post_processing['tblite0_chem_pot_beta']
            
            gap_alpha = post_processing['tblite0_gap_alpha']
            gap_beta = post_processing['tblite0_gap_beta']
        
            hoao_a = post_processing['tblite0_HOAO_alpha']
            hoao_b = post_processing['tblite0_HOAO_beta']
        
            luao_a = post_processing['tblite0_LUAO_alpha']
            luao_b = post_processing['tblite0_LUAO_beta']

            V_f_final_list = []
            V_f_final_list.append(charges)
            V_f_final_list += [d_density_alpha, p_density_alpha, s_density_alpha, 
                                   d_density_beta, p_density_beta, s_density_beta]
            V_f_final_list += [chem_pot_alpha, chem_pot_beta]

            V_f_final_list += [gap_alpha, gap_beta]
                
            V_f_final_list += [hoao_a, hoao_b]
                
            V_f_final_list += [luao_a, luao_b]
        
            V_f_final = np.column_stack(V_f_final_list)
            
            charge_one_hot = list(np.eye(8)[int(x_d[0] + 4)])
            spin_one_hot = list(np.eye(6)[int(x_d[1] * 2)])
            fe_spin_one_hot = list(np.eye(6)[int(x_d[2] * 2)])
        
            iron_index = 0
            for i, a in enumerate(mol.GetAtoms()):
                if a.GetAtomicNum() == 26:
                    iron_index = i
                    break
                    
            quadrupole = [post_processing[f'tblite0_qm_A_{x+13*' '}_alpha'][iron_index] + 
                          post_processing[f'tblite0_qm_A_{x+13*' '}_beta'][iron_index] 
                          for x in ['zz','yy','xx','xy','xz','yz']] + \
                         [post_processing[f'tblite0_ext_qm_e_{x+9*' '}_alpha'][iron_index] + 
                          post_processing[f'tblite0_ext_qm_e_{x+9*' '}_beta'][iron_index] 
                          for x in ['zz','yy','xx','xy','xz','yz']] + \
                         [post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_alpha'][iron_index] + 
                          post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_beta'][iron_index] 
                          for x in ['zz','yy','xx','xy','xz','yz']] + \
                         [post_processing[f'tblite0_qm_A_alpha'][iron_index], 
                          post_processing[f'tblite0_qm_A_beta'][iron_index],
                          post_processing[f'tblite0_ext_qm_e_alpha'][iron_index],
                          post_processing[f'tblite0_ext_qm_e_beta'][iron_index],
                          post_processing[f'tblite0_ext_qm_Z_alpha'][iron_index],
                          post_processing[f'tblite0_ext_qm_Z_beta'][iron_index]]
        
            threed_list = list(Descriptors3D.CalcMolDescriptors3D(mol).values())

            x_d_final_list = []

            x_d_final_list += charge_one_hot
            x_d_final_list += spin_one_hot
            x_d_final_list += fe_spin_one_hot
            x_d_final_list += threed_list
            x_d_final_list += quadrupole
            x_d_final_list += [x_d[3]]
            x_d_final_list += [charges[iron_index]]
            x_d_final = np.asarray(x_d_final_list)
            
            x_dsize = x_d_final.shape[0]
            datapoint_list.append(data.MoleculeDatapoint(mol, y, x_d=x_d_final, V_f=V_f_final, name=file))
        
        all_data = np.asarray(datapoint_list)
        
        kf = KFold(n_splits=10,shuffle=True,random_state=j)
        
        total_r2_is = 0
        total_r2_qs = 0
        total_mse_is = 0
        total_mse_qs = 0
        total_mae_is = 0
        total_mae_qs = 0
        
        for i, (train_index, test_index) in enumerate(kf.split(all_data)):
            train_data = all_data[train_index]
            test_data = all_data[test_index]
        
            featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(
                                            atom_featurizer=TMCAtomFeaturizer.complex(mask=mask_atom),
                                            bond_featurizer=TMCBondFeaturizer(mask=mask_bond),
                                            extra_atom_fdim=V_f_final.shape[1])
            
            train_dset = data.MoleculeDataset(train_data, featurizer)
            targets_scaler = train_dset.normalize_targets()
            X_d_scaler = train_dset.normalize_inputs("X_d")
            V_f_scaler = train_dset.normalize_inputs("V_f")
            
            val_dset = data.MoleculeDataset(test_data, featurizer)
            val_dset.normalize_targets(targets_scaler)
            val_dset.normalize_inputs("X_d", X_d_scaler)
            val_dset.normalize_inputs("V_f", V_f_scaler)
            
            test_dset = data.MoleculeDataset(test_data, featurizer)
            test_dset.cache = True
            
            train_loader = data.build_dataloader(train_dset, num_workers=num_workers, batch_size=100)
            val_loader = data.build_dataloader(val_dset, num_workers=num_workers, shuffle=False)
            test_loader = data.build_dataloader(test_dset, num_workers=num_workers, shuffle=False)
            
            n_V_features = featurizer.atom_fdim - featurizer.extra_atom_fdim
            V_f_transform = ScaleTransform.from_standard_scaler(V_f_scaler, pad=n_V_features)
            graph_transform = GraphTransform(V_f_transform, ScaleTransform([0],[1]))
            
            X_d_transform = ScaleTransform.from_standard_scaler(X_d_scaler)
            
            output_transform = UnscaleTransform.from_standard_scaler(targets_scaler)
            
            mp = nn.BondMessagePassing(d_h=800, depth=4,
                                       d_v=featurizer.atom_fdim, 
                                       d_e=featurizer.bond_fdim, 
                                       graph_transform=graph_transform,
                                       dropout=0.2,
                                      )
            agg = nn.AttentiveAggregation(output_size=800)
            ffn = nn.RegressionFFN(
                input_dim=mp.output_dim + x_dsize,
                n_tasks=len(target_columns),
                output_transform=output_transform,
                hidden_dim=400,
                n_layers=3,
                dropout=0.2,
                criterion=nn.metrics.MAE()
            )
            
            batch_norm = True
            metric_list = [nn.metrics.MAE(), nn.metrics.MSE(), nn.metrics.R2Score()]
            mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list, X_d_transform=X_d_transform, init_lr=0.0002)
            early_stop_callback = EarlyStopping(monitor="val_loss", 
                                                min_delta=0.00, 
                                                patience=400, 
                                                verbose=False, 
                                                mode="min",
                                                stopping_threshold=0.22)
            
            trainer = pl.Trainer(
                logger=True,
                enable_checkpointing=True,
                enable_progress_bar=True,
                accelerator="gpu",
                devices=1,
                precision="32-true",
                gradient_clip_val=0.01,
                gradient_clip_algorithm="norm",
                max_epochs=200,
            )
            
            trainer.fit(mpnn, train_loader, None)#val_loader)
            
            trainer = pl.Trainer(
                logger=True,
                enable_checkpointing=True,
                enable_progress_bar=True,
                accelerator="gpu",
                devices=1,
                max_epochs=1,
            )
            
            out = [trainer.predict(mpnn, x, return_predictions=True) for x in [test_loader]]
            
            out_is = [torch.tensor([y for x in o for y in x[:, 0]]).cpu().detach().numpy() for o in out]
            actual_is = [torch.tensor([d.y[0] for d in data]).cpu().detach().numpy() for data in [test_data]]
            
            mae = [abs(a_is - o_is).sum() / a_is.shape[0] for a_is, o_is in zip(actual_is, out_is)]
            mse = [((a_is - o_is)**2).sum() / a_is.shape[0] for a_is, o_is in zip(actual_is, out_is)]
            r2 = [r2_score(a_is, o_is) for a_is, o_is in zip(actual_is, out_is)]

            total_mae_is += mae[0]
            total_mse_is += mse[0]
            total_r2_is += r2[0]
            
            print(f"No {j}, i = {i}, IS TEST: MAE = {mae[0]}, MSE = {mse[0]}, R2 = {r2[0]}",file=f)
            
            out_qs = [torch.tensor([y for x in o for y in x[:, 1]]).cpu().detach().numpy() for o in out]
            actual_qs = [torch.tensor([d.y[1] for d in data]).cpu().detach().numpy() for data in [test_data]]
            
            mae = [abs(a_qs - o_qs).sum() / a_qs.shape[0] for a_qs, o_qs in zip(actual_qs, out_qs)]
            mse = [((a_qs - o_qs)**2).sum() / a_qs.shape[0] for a_qs, o_qs in zip(actual_qs, out_qs)]
            r2 = [r2_score(a_qs, o_qs) for a_qs, o_qs in zip(actual_qs, out_qs)]
        
            total_mae_qs += mae[0]
            total_mse_qs += mse[0]
            total_r2_qs += r2[0]
            
            print(f"No {j}, i = {i}, QS TEST: MAE = {mae[0]}, MSE = {mse[0]}, R2 = {r2[0]}", file=f)
        
        print(f"No {j}, Validation: IS_MAE = {total_mae_is/10}, IS_MSE = {total_mse_is/10}, IS_R2 = {total_r2_is/10} | QS_MAE = {total_mae_qs/10}, QS_MSE = {total_mse_qs/10}, QS_R2 = {total_r2_qs/10}", file=f)

### 3. Ablation study

In [ ]:
import torch
torch.set_float32_matmul_precision('high')
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

from chemprop.nn.transforms import ScaleTransform, GraphTransform, UnscaleTransform

from featurizers.atom import TMCAtomFeaturizer
from featurizers.bond import TMCBondFeaturizer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from rdkit.Chem import Descriptors3D

import pandas as pd
import numpy as np
from pathlib import Path

from lightning import pytorch as pl

from chemprop import data, featurizers, models, nn

#train_data, val_test_data = train_test_split(all_data, test_size=0.2, random_state=42)
#val_data, test_data = train_test_split(val_test_data, test_size=0.5, random_state=42)

keep_list = ['charges', 'densities', 'chemp_pot', 'gap', 'hoao', 'luao', 'charge_one_hot', 'spin_one_hot', 'fe_spin_one_hot', 'threed_list', 'quadrupole', 'temp', 'charge_iron']
mask_atom = [True for _ in range(8)]
mask_bond = [True for _ in range(4)]

with open("ablation.txt", "w") as f:
        
    num_workers = 10 # number of workers for dataloader. 0 means using main process for data loading
    target_columns = ['IS', 'QS'] # classification of activity (either 0 or 1)
    feature_columns = ['CHARGE', 'SPIN', 'FE SPIN', 'TEMP'] # name of feature columns in csv
    
    ys = smiles_csv.loc[:, target_columns].values
    x_ds = smiles_csv.loc[:, feature_columns].values
    x_dsize = 0
    
    datapoint_list = []
    for mol, file, y, x_d in zip(mol_list, file_list, ys, x_ds):
        desc_path = osp.join(data_folder, file[:-4], 'tblite-data.npz')
        post_processing = np.load(desc_path)
    
        charges = post_processing['tblite0_q_A_alpha'] + post_processing['tblite0_q_A_beta']
    
        d_density_alpha = post_processing['tblite0_p_d_alpha']
        p_density_alpha = post_processing['tblite0_p_p_alpha']
        s_density_alpha = post_processing['tblite0_p_s_alpha']
    
        d_density_beta = post_processing['tblite0_p_d_beta']
        p_density_beta = post_processing['tblite0_p_p_beta']
        s_density_beta = post_processing['tblite0_p_s_beta']
    
        chem_pot_alpha = post_processing['tblite0_chem_pot_alpha']
        chem_pot_beta = post_processing['tblite0_chem_pot_beta']
        
        gap_alpha = post_processing['tblite0_gap_alpha']
        gap_beta = post_processing['tblite0_gap_beta']
    
        hoao_a = post_processing['tblite0_HOAO_alpha']
        hoao_b = post_processing['tblite0_HOAO_beta']
    
        luao_a = post_processing['tblite0_LUAO_alpha']
        luao_b = post_processing['tblite0_LUAO_beta']

        V_f_final_list = []
        
        if 'charges' in keep_list:
            V_f_final_list.append(charges)

        if 'densities' in keep_list:
            V_f_final_list += [d_density_alpha, p_density_alpha, s_density_alpha, 
                               d_density_beta, p_density_beta, s_density_beta]
            
        if 'chemp_pot' in keep_list:
            V_f_final_list += [chem_pot_alpha, chem_pot_beta]
            
        if 'gap' in keep_list:
            V_f_final_list += [gap_alpha, gap_beta]
            
        if 'hoao' in keep_list:
            V_f_final_list += [hoao_a, hoao_b]
            
        if 'luao' in keep_list:
            V_f_final_list += [luao_a, luao_b]
    
        V_f_final = np.column_stack(V_f_final_list if len(V_f_final_list) != 0 else np.zeros_like(charges))
        
        charge_one_hot = list(np.eye(8)[int(x_d[0] + 4)])
        spin_one_hot = list(np.eye(6)[int(x_d[1] * 2)])
        fe_spin_one_hot = list(np.eye(6)[int(x_d[2] * 2)])
    
        iron_index = 0
        for i, a in enumerate(mol.GetAtoms()):
            if a.GetAtomicNum() == 26:
                iron_index = i
                break
                
        quadrupole = [post_processing[f'tblite0_qm_A_{x+13*' '}_alpha'][iron_index] + 
                      post_processing[f'tblite0_qm_A_{x+13*' '}_beta'][iron_index] 
                      for x in ['zz','yy','xx','xy','xz','yz']] + \
                     [post_processing[f'tblite0_ext_qm_e_{x+9*' '}_alpha'][iron_index] + 
                      post_processing[f'tblite0_ext_qm_e_{x+9*' '}_beta'][iron_index] 
                      for x in ['zz','yy','xx','xy','xz','yz']] + \
                     [post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_alpha'][iron_index] + 
                      post_processing[f'tblite0_ext_qm_Z_{x+9*' '}_beta'][iron_index] 
                      for x in ['zz','yy','xx','xy','xz','yz']] + \
                     [post_processing[f'tblite0_qm_A_alpha'][iron_index], 
                      post_processing[f'tblite0_qm_A_beta'][iron_index],
                      post_processing[f'tblite0_ext_qm_e_alpha'][iron_index],
                      post_processing[f'tblite0_ext_qm_e_beta'][iron_index],
                      post_processing[f'tblite0_ext_qm_Z_alpha'][iron_index],
                      post_processing[f'tblite0_ext_qm_Z_beta'][iron_index]]
    
        threed_list = list(Descriptors3D.CalcMolDescriptors3D(mol).values())

        x_d_final_list = []

        if 'charge_one_hot' in keep_list:
            x_d_final_list += charge_one_hot

        if 'spin_one_hot' in keep_list:
            x_d_final_list += spin_one_hot

        if 'fe_spin_one_hot' in keep_list:
            x_d_final_list += fe_spin_one_hot

        if 'threed_list' in keep_list:
            x_d_final_list += threed_list

        if 'quadrupole' in keep_list:
            x_d_final_list += quadrupole

        if 'temp' in keep_list:
            x_d_final_list += [x_d[3]]

        if 'charge_iron' in keep_list:
            x_d_final_list += [charges[iron_index]]
        
        x_d_final = np.asarray(x_d_final_list if len(x_d_final_list) != 0 else [0])
        
        x_dsize = x_d_final.shape[0]
        datapoint_list.append(data.MoleculeDatapoint(mol, y, x_d=x_d_final, V_f=V_f_final, name=file))
    
    all_data = np.asarray(datapoint_list)
    
    kf = KFold(n_splits=10,shuffle=True,random_state=j)
    
    total_r2_is = 0
    total_r2_qs = 0
    total_mse_is = 0
    total_mse_qs = 0
    total_mae_is = 0
    total_mae_qs = 0
    
    for i, (train_index, test_index) in enumerate(kf.split(all_data)):
        train_data = all_data[train_index]
        test_data = all_data[test_index]
    
        featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(
                                        atom_featurizer=TMCAtomFeaturizer.complex(mask=mask_atom),
                                        bond_featurizer=TMCBondFeaturizer(mask=mask_bond),
                                        extra_atom_fdim=V_f_final.shape[1])
        
        train_dset = data.MoleculeDataset(train_data, featurizer)
        targets_scaler = train_dset.normalize_targets()
        X_d_scaler = train_dset.normalize_inputs("X_d")
        V_f_scaler = train_dset.normalize_inputs("V_f")
        
        val_dset = data.MoleculeDataset(test_data, featurizer)
        val_dset.normalize_targets(targets_scaler)
        val_dset.normalize_inputs("X_d", X_d_scaler)
        val_dset.normalize_inputs("V_f", V_f_scaler)
        
        test_dset = data.MoleculeDataset(test_data, featurizer)
        test_dset.cache = True
        
        train_loader = data.build_dataloader(train_dset, num_workers=num_workers, batch_size=100)
        val_loader = data.build_dataloader(val_dset, num_workers=num_workers, shuffle=False)
        test_loader = data.build_dataloader(test_dset, num_workers=num_workers, shuffle=False)
        
        n_V_features = featurizer.atom_fdim - featurizer.extra_atom_fdim
        V_f_transform = ScaleTransform.from_standard_scaler(V_f_scaler, pad=n_V_features)
        graph_transform = GraphTransform(V_f_transform, ScaleTransform([0],[1]))
        
        X_d_transform = ScaleTransform.from_standard_scaler(X_d_scaler)
        
        output_transform = UnscaleTransform.from_standard_scaler(targets_scaler)
        
        mp = nn.BondMessagePassing(d_h=800, depth=4,
                                   d_v=featurizer.atom_fdim, 
                                   d_e=featurizer.bond_fdim, 
                                   graph_transform=graph_transform,
                                   dropout=0.2,
                                  )
        agg = nn.AttentiveAggregation(output_size=800)
        ffn = nn.RegressionFFN(
            input_dim=mp.output_dim + x_dsize,
            n_tasks=len(target_columns),
            output_transform=output_transform,
            hidden_dim=400,
            n_layers=3,
            dropout=0.2,
            criterion=nn.metrics.MAE()
        )
        
        batch_norm = True
        metric_list = [nn.metrics.MAE(), nn.metrics.MSE(), nn.metrics.R2Score()]
        mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list, X_d_transform=X_d_transform, init_lr=0.0002)
        early_stop_callback = EarlyStopping(monitor="val_loss", 
                                            min_delta=0.00, 
                                            patience=400, 
                                            verbose=False, 
                                            mode="min",
                                            stopping_threshold=0.22)
        
        trainer = pl.Trainer(
            logger=True,
            enable_checkpointing=True,
            enable_progress_bar=True,
            accelerator="gpu",
            devices=1,
            precision="32-true",
            gradient_clip_val=0.01,
            gradient_clip_algorithm="norm",
            max_epochs=200,
        )
        
        trainer.fit(mpnn, train_loader, None)#val_loader)
        
        trainer = pl.Trainer(
            logger=True,
            enable_checkpointing=True,
            enable_progress_bar=True,
            accelerator="gpu",
            devices=1,
            max_epochs=1,
        )
        
        out = [trainer.predict(mpnn, x, return_predictions=True) for x in [test_loader]]
        
        out_is = [torch.tensor([y for x in o for y in x[:, 0]]).cpu().detach().numpy() for o in out]
        actual_is = [torch.tensor([d.y[0] for d in data]).cpu().detach().numpy() for data in [test_data]]
        
        mae = [abs(a_is - o_is).sum() / a_is.shape[0] for a_is, o_is in zip(actual_is, out_is)]
        mse = [((a_is - o_is)**2).sum() / a_is.shape[0] for a_is, o_is in zip(actual_is, out_is)]
        r2 = [r2_score(a_is, o_is) for a_is, o_is in zip(actual_is, out_is)]

        total_mae_is += mae[0]
        total_mse_is += mse[0]
        total_r2_is += r2[0]
        
        print(f"No {j}, i = {i}, IS TEST: MAE = {mae[0]}, MSE = {mse[0]}, R2 = {r2[0]}",file=f)
        
        out_qs = [torch.tensor([y for x in o for y in x[:, 1]]).cpu().detach().numpy() for o in out]
        actual_qs = [torch.tensor([d.y[1] for d in data]).cpu().detach().numpy() for data in [test_data]]
        
        mae = [abs(a_qs - o_qs).sum() / a_qs.shape[0] for a_qs, o_qs in zip(actual_qs, out_qs)]
        mse = [((a_qs - o_qs)**2).sum() / a_qs.shape[0] for a_qs, o_qs in zip(actual_qs, out_qs)]
        r2 = [r2_score(a_qs, o_qs) for a_qs, o_qs in zip(actual_qs, out_qs)]
    
        total_mae_qs += mae[0]
        total_mse_qs += mse[0]
        total_r2_qs += r2[0]
        
        print(f"No {j}, i = {i}, QS TEST: MAE = {mae[0]}, MSE = {mse[0]}, R2 = {r2[0]}", file=f)
        
        print(f"Validation: IS_MAE = {total_mae_is/10}, IS_MSE = {total_mse_is/10}, IS_R2 = {total_r2_is/10} | QS_MAE = {total_mae_qs/10}, QS_MSE = {total_mse_qs/10}, QS_R2 = {total_r2_qs/10}", file=f)